# GBDT vs NN

Thin orchestration notebook for the default `NN` vs `GBDT` analysis.

In [16]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [ ]:
from mfa import load_config, run_analysis

config = load_config(
    project_dir / "configs" / "paper_modern_tabpfn_new_vs_tabicl_new.yaml"
)
result = run_analysis(
    config,
    # datasets=[
    #     "Bank_Customer_Churn",
    #     "diabetes",
    #     "miami_housing",
    #     "physiochemical_protein",
    #     "website_phishing",
    # ],
)
result.analysis_table.head()

09:10:15 INFO mfa.pipeline: Starting analysis: comparisons=tabpfn_new_vs_tabicl_new; scope=all benchmark datasets; unit=dataset; method_variant=default,tuned; n_jobs=32
09:10:15 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (34110 rows, 51 dataset(s))


09:10:15 INFO mfa.pipeline: Stage 2/5 meta-features: cache hit (816 rows, 51 dataset(s))
09:10:15 INFO mfa.pipeline: Stage 3/5 pairwise gaps: cache hit (816 rows, 51 dataset(s))
09:10:15 INFO mfa.pipeline: Stage 4/5 analysis table: joining and aggregating at dataset level
/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/aggregation.py:150: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_level = analysis.groupby(GROUP_COLUMNS, as_index=False, dropna=False).agg(**aggregations)
/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/aggregation.py:150: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem,irregularity
0,APSFailure,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,TabPFN new,TabICL new,NaN,9,50666.666667,170.0,...,0.481520,0.005776,0.000000,0.001537,0.000573,0.000191,0.481520,0.168284,0.056095,2.708453
1,Amazon_employee_access,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,TabPFN new,TabICL new,NaN,9,21846.000000,9.0,...,0.801420,0.146974,0.789150,0.000351,0.003778,0.001259,0.012270,0.099236,0.033079,NaN
2,Another-Dataset-on-used-Fiat-500,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,TabPFN new,TabICL new,NaN,30,1025.333333,7.0,...,0.391746,715.812974,0.081010,10.645720,6.555674,1.196897,0.310735,0.191229,0.034913,-0.021913
3,Bank_Customer_Churn,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,TabPFN new,TabICL new,NaN,9,6666.666667,10.0,...,0.240931,0.128107,0.265143,-0.000320,0.001064,0.000355,-0.024211,0.094995,0.031665,-0.759189
4,Bioresponse,tabpfn_new_vs_tabicl_new,tabpfn_new,tabicl_new,TabPFN new,TabICL new,NaN,9,2500.666667,1776.0,...,0.153651,0.138276,1.000000,-0.015955,0.003924,0.001308,-0.846349,0.146674,0.048891,-0.424214


In [18]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        521ca0dcd7b2aa2d
comparison_name:    tabpfn_new_vs_tabicl_new
analysis_table:     (51, 204)
gap_table:          (816, 17)
metafeature_table:  (816, 188)
correlation_results: 186 features tested
correction_result:  bh
multivariate_result: None


## Correlation summary table

In [19]:
import numpy as np

# Build a comprehensive table from correlation + correction results
corr_df = pd.DataFrame([r.__dict__ for r in result.correlation_results])

if result.correction_result is not None:
    corr_df["p_value_adj"] = result.correction_result.adjusted_p_values
    corr_df["rejected"] = result.correction_result.rejected

# Add a significance star column for quick scanning
p_col = "p_value_adj" if "p_value_adj" in corr_df.columns else "p_value"
corr_df["sig"] = np.where(
    corr_df[p_col] < 0.001,
    "***",
    np.where(corr_df[p_col] < 0.01, "**", np.where(corr_df[p_col] < 0.05, "*", "")),
)

display_cols = [
    "predictor",
    "statistic",
    "ci_lower",
    "ci_upper",
    "p_value",
    *(["p_value_adj", "rejected"] if "p_value_adj" in corr_df.columns else []),
    "sig",
    "n_observations",
    "direction_confirmed",
]

corr_df[display_cols].sort_values("p_value")

,predictor,statistic,ci_lower,ci_upper,p_value,p_value_adj,rejected,sig,n_observations,direction_confirmed
141,pymfe__attr_conc.median,0.505018,0.241738,0.702700,0.000184,0.034005,True,*,50,None
67,pymfe__h_mean.mean,0.430930,0.062418,0.730419,0.013806,0.494158,False,,32,None
142,pymfe__attr_conc.min,0.339928,0.043648,0.586208,0.015724,0.494158,False,,50,None
70,pymfe__h_mean.sd,0.419189,0.051354,0.717648,0.016935,0.494158,False,,32,None
157,pymfe__class_conc.median,0.369369,0.053456,0.632168,0.024455,0.494158,False,,37,None
...,...,...,...,...,...,...,...,...,...,...
28,outlier_fraction_iqr,0.009189,-0.277326,0.289614,0.951671,0.967357,False,,46,None
139,pymfe__attr_conc.max,-0.004178,-0.285026,0.270810,0.977029,0.983621,False,,50,None
104,pymfe__min.min,0.004441,-0.319163,0.315310,0.978304,0.983621,False,,40,None
19,cat_cardinality_to_n_ratio,-0.000153,-0.344793,0.352652,0.999316,0.999316,False,,34,None


## Correlation scatter plots

In [20]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="ticks", context="talk")

analysis = result.analysis_table.copy()
predictors = corr_df["predictor"].tolist()
n_preds = len(predictors)
ncols = min(4, n_preds)
nrows = int(np.ceil(n_preds / ncols))

fig, axes = plt.subplots(
    nrows, ncols, figsize=(5.5 * ncols, 5 * nrows), sharey=True, squeeze=False
)

for idx, predictor in enumerate(predictors):
    ax = axes[idx // ncols][idx % ncols]
    plot_df = analysis[[predictor, "delta_norm"]].dropna()

    # Use log scale for features that are strictly positive and span orders of magnitude
    use_log = (plot_df[predictor] > 0).all() and (
        plot_df[predictor].max() / plot_df[predictor].min() > 10
    )

    sns.scatterplot(data=plot_df, x=predictor, y="delta_norm", s=35, alpha=0.85, ax=ax)
    if use_log:
        ax.set_xscale("log")

    ax.set_xlabel(predictor)
    if idx % ncols == 0:
        ax.set_ylabel(r"$\Delta_{\ell\ell}$ (NN − GBDT)")
    else:
        ax.set_ylabel("")

    # Annotate with correlation and adjusted p-value
    row = corr_df.loc[corr_df["predictor"] == predictor].iloc[0]
    p_display = row.get("p_value_adj", row["p_value"])
    ax.text(
        0.05,
        0.95,
        f"r = {row['statistic']:.3f}\np = {p_display:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        bbox={"facecolor": "#d9d9d9", "edgecolor": "#9e9e9e", "alpha": 0.9},
    )

# Hide unused subplots
for idx in range(n_preds, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

sns.despine()
fig.tight_layout()
plt.show()